# FinMark Data Preprocessing
This notebook contains preprocessing steps for Customer Segments, Transactions, and Social Media datasets.

In [2]:
import pandas as pd
import numpy as np

In [3]:
segments = pd.read_csv("customer_demographics_contaminated.csv")
transactions = pd.read_csv("customer_transactions_contaminated.csv")
social = pd.read_csv("social_media_interactions_contaminated.csv")

In [4]:
print("Segments shape:", segments.shape)
print(segments.head())

print("\nTransactions shape:", transactions.shape)
print(transactions.head())

print("\nSocial shape:", social.shape)
print(social.head())

Segments shape: (3200, 6)
                             CustomerID   Age  Gender           Location  \
0  9207fa75-5758-48d1-94ad-19c041e0520f  51.0  Female         Jensenberg   
1  5fb09cd8-a473-46f7-80bd-6e49cf509078   NaN  Female       Castilloport   
2  c139496e-cc89-498a-bd90-1fb4627b6cff  37.0    Male  Lake Jennifertown   
3  50118139-7264-428f-81cc-a25fddc5d6dd  44.0    Male          Port Carl   
4  7d1f2bbc-8d16-4fbc-9b37-ece3324e8ed4  50.0  Female          Jessebury   

  IncomeLevel  SignupDate  
0         Low  2022-11-17  
1        High  2020-07-21  
2         NaN  2021-01-01  
3      Medium  2024-06-10  
4        High  2023-08-24  

Transactions shape: (3200, 6)
                             CustomerID                         TransactionID  \
0  60567026-f719-4cd6-849e-137e86d8938f  5ff75116-0a50-4d04-80fb-31e5ccbb0769   
1  4090ba85-b111-4f75-a792-c777965f5255  2c39b9fe-ff57-4d39-9321-9f5cdf187aa1   
2  9223891b-73ff-4d5c-b8ae-13ece82ee28b  f79588dd-3db9-4ffa-97f8-7de0e64259

In [5]:
segments.info()
transactions.info()
social.info()

<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   CustomerID   3200 non-null   str  
 1   Age          2909 non-null   str  
 2   Gender       3200 non-null   str  
 3   Location     3200 non-null   str  
 4   IncomeLevel  2897 non-null   str  
 5   SignupDate   3200 non-null   str  
dtypes: str(6)
memory usage: 150.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CustomerID       3200 non-null   str  
 1   TransactionID    3200 non-null   str  
 2   TransactionDate  3200 non-null   str  
 3   Amount           2896 non-null   str  
 4   ProductCategory  2901 non-null   str  
 5   PaymentMethod    3200 non-null   str  
dtypes: str(6)
memory usage: 150.1 KB
<class 'pandas.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199

In [6]:
print("Segments duplicate rows:", segments.duplicated().sum())
print("Transactions duplicate rows:", transactions.duplicated().sum())
print("Social duplicate rows:", social.duplicated().sum())

# Full-row duplicate counts alone are misleading here: some records share the
# same business key (ID) but differ slightly, so check ID-based duplicates too.
print("\nSegments duplicate CustomerID:", segments['CustomerID'].duplicated().sum())
print("Transactions duplicate TransactionID:", transactions['TransactionID'].duplicated().sum())
print("Social duplicate InteractionID:", social['InteractionID'].duplicated().sum())

print("\nSegments missing:\n", segments.isnull().sum())
print("\nTransactions missing:\n", transactions.isnull().sum())
print("\nSocial missing:\n", social.isnull().sum())

Segments duplicate rows: 177
Transactions duplicate rows: 185
Social duplicate rows: 180

Segments duplicate CustomerID: 200
Transactions duplicate TransactionID: 200
Social duplicate InteractionID: 200

Segments missing:
 CustomerID       0
Age            291
Gender           0
Location         0
IncomeLevel    303
SignupDate       0
dtype: int64

Transactions missing:
 CustomerID           0
TransactionID        0
TransactionDate      0
Amount             304
ProductCategory    299
PaymentMethod        0
dtype: int64

Social missing:
 CustomerID           0
InteractionID        0
InteractionDate      0
Platform           311
InteractionType      0
Sentiment          329
dtype: int64


In [7]:
# dtypes alone don't reveal contamination hiding inside object/string columns, so inspect values directly
age_numeric = pd.to_numeric(segments['Age'], errors='coerce')
print("Age = 'Unknown':", (segments['Age'] == 'Unknown').sum())
print("Age out-of-range (<=0 or >100):", ((age_numeric <= 0) | (age_numeric > 100)).sum())
print("IncomeLevel categories:", segments['IncomeLevel'].unique())

amount_numeric = pd.to_numeric(transactions['Amount'], errors='coerce')
print("\nAmount non-numeric values:", transactions.loc[amount_numeric.isnull() & transactions['Amount'].notnull(), 'Amount'].unique())
print("Amount negative values:", (amount_numeric < 0).sum())
print("ProductCategory categories:", transactions['ProductCategory'].unique())

print("\nPlatform categories:", social['Platform'].unique())
print("Sentiment categories:", social['Sentiment'].unique())

# Dates fail to parse with a single format when the column mixes YYYY-MM-DD and DD/MM/YYYY
print("\nUnparseable TransactionDate (ISO-only parse):", pd.to_datetime(transactions['TransactionDate'], format='%Y-%m-%d', errors='coerce').isnull().sum())
print("Unparseable InteractionDate (ISO-only parse):", pd.to_datetime(social['InteractionDate'], format='%Y-%m-%d', errors='coerce').isnull().sum())
print("Unparseable SignupDate (ISO-only parse):", pd.to_datetime(segments['SignupDate'], format='%Y-%m-%d', errors='coerce').isnull().sum())

Age = 'Unknown': 34
Age out-of-range (<=0 or >100): 63
IncomeLevel categories: <StringArray>
['Low', 'High', nan, 'Medium']
Length: 4, dtype: str

Amount non-numeric values: <StringArray>
['Free']
Length: 1, dtype: str
Amount negative values: 40
ProductCategory categories: <StringArray>
[       'Clothing', 'Health & Beauty',      'Automotive',   'Home & Garden',
               nan,     'Electronics']
Length: 6, dtype: str

Platform categories: <StringArray>
[nan, 'Twitter', 'Instagram', 'Facebook']
Length: 4, dtype: str
Sentiment categories: <StringArray>
[nan, 'Neutral', 'Positive', 'Negative', 'Very Negative', 'Very Positive']
Length: 6, dtype: str

Unparseable TransactionDate (ISO-only parse): 98
Unparseable InteractionDate (ISO-only parse): 100
Unparseable SignupDate (ISO-only parse): 98


In [8]:
# Helper: SignupDate/TransactionDate/InteractionDate mix YYYY-MM-DD and DD/MM/YYYY strings
def parse_mixed_dates(series):
    iso = pd.to_datetime(series, format='%Y-%m-%d', errors='coerce')
    dmy = pd.to_datetime(series, format='%d/%m/%Y', errors='coerce')
    return iso.fillna(dmy)

In [9]:
# --- Clean Segments: dedupe (row + business key), fix Age, handle missing categories, parse dates ---
segments = segments.drop_duplicates()
segments = segments.drop_duplicates(subset='CustomerID', keep='first')

segments['Age'] = segments['Age'].replace('Unknown', np.nan)
segments['Age'] = pd.to_numeric(segments['Age'], errors='coerce')
segments.loc[(segments['Age'] <= 0) | (segments['Age'] > 100), 'Age'] = np.nan
segments['Age'] = segments['Age'].fillna(segments['Age'].median())

segments['Gender'] = segments['Gender'].str.strip()
# Missing IncomeLevel is ~10% of rows; keep the rows and flag as its own category instead of dropping them
segments['IncomeLevel'] = segments['IncomeLevel'].fillna('Unknown')

segments['SignupDate'] = parse_mixed_dates(segments['SignupDate'])
segments['CustomerID'] = segments['CustomerID'].astype(str).str.strip()

In [10]:
# --- Clean Transactions: dedupe (row + business key), fix Amount, handle missing categories, parse dates ---
transactions = transactions.drop_duplicates()
transactions = transactions.drop_duplicates(subset='TransactionID', keep='first')

transactions['Amount'] = transactions['Amount'].replace('Free', 0)
transactions['Amount'] = pd.to_numeric(transactions['Amount'], errors='coerce')
transactions.loc[transactions['Amount'] < 0, 'Amount'] = np.nan
# Median is more robust than mean here since a handful of contaminated values skew the distribution
transactions['Amount'] = transactions['Amount'].fillna(transactions['Amount'].median())

transactions['ProductCategory'] = transactions['ProductCategory'].fillna('Unknown')
transactions['PaymentMethod'] = transactions['PaymentMethod'].str.strip()

transactions['TransactionDate'] = parse_mixed_dates(transactions['TransactionDate'])
transactions['CustomerID'] = transactions['CustomerID'].astype(str).str.strip()
transactions['TransactionID'] = transactions['TransactionID'].astype(str).str.strip()

In [11]:
# --- Clean Social: dedupe (row + business key), handle missing categories, parse dates ---
social = social.drop_duplicates()
social = social.drop_duplicates(subset='InteractionID', keep='first')

social['Platform'] = social['Platform'].fillna('Unknown')
social['Sentiment'] = social['Sentiment'].fillna('Unknown')
social['InteractionType'] = social['InteractionType'].str.strip()

social['InteractionDate'] = parse_mixed_dates(social['InteractionDate'])
social['CustomerID'] = social['CustomerID'].astype(str).str.strip()
social['InteractionID'] = social['InteractionID'].astype(str).str.strip()

In [12]:
# --- Post-cleaning verification: confirm dedupe/missing-value/dtype fixes took effect ---
print("Duplicate rows -> Segments:", segments.duplicated().sum(), "Transactions:", transactions.duplicated().sum(), "Social:", social.duplicated().sum())
print("Duplicate IDs -> CustomerID:", segments['CustomerID'].duplicated().sum(),
      "TransactionID:", transactions['TransactionID'].duplicated().sum(),
      "InteractionID:", social['InteractionID'].duplicated().sum())

print("\nSegments missing:\n", segments.isnull().sum())
print("\nTransactions missing:\n", transactions.isnull().sum())
print("\nSocial missing:\n", social.isnull().sum())

segments.info()
transactions.info()
social.info()

Duplicate rows -> Segments: 0 Transactions: 0 Social: 0
Duplicate IDs -> CustomerID: 0 TransactionID: 0 InteractionID: 0

Segments missing:
 CustomerID     0
Age            0
Gender         0
Location       0
IncomeLevel    0
SignupDate     0
dtype: int64

Transactions missing:
 CustomerID         0
TransactionID      0
TransactionDate    0
Amount             0
ProductCategory    0
PaymentMethod      0
dtype: int64

Social missing:
 CustomerID         0
InteractionID      0
InteractionDate    0
Platform           0
InteractionType    0
Sentiment          0
dtype: int64
<class 'pandas.DataFrame'>
Index: 3000 entries, 0 to 2999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   CustomerID   3000 non-null   str           
 1   Age          3000 non-null   float64       
 2   Gender       3000 non-null   str           
 3   Location     3000 non-null   str           
 4   IncomeLevel  3000 non-null   str   

In [13]:
segments.to_csv("Cleaned_Customer_Demographics.csv", index=False)
transactions.to_csv("Cleaned_Customer_Transactions.csv", index=False)
social.to_csv("Cleaned_Social_Media_Interactions.csv", index=False)

print("Saved: Cleaned_Customer_Demographics.csv, Cleaned_Customer_Transactions.csv, Cleaned_Social_Media_Interactions.csv")

Saved: Cleaned_Customer_Demographics.csv, Cleaned_Customer_Transactions.csv, Cleaned_Social_Media_Interactions.csv


## Preprocessing Summary

**Steps performed**
- Removed exact duplicate rows, then removed remaining duplicates by business key (`CustomerID`, `TransactionID`, `InteractionID`) since some duplicate records were not byte-identical.
- Converted placeholder contamination (`"Unknown"` in `Age`, `"Free"` in `Amount`) to proper missing/numeric values before imputing.
- Treated impossible values as missing: `Age` <= 0 or > 100, `Amount` < 0.
- Imputed missing numeric fields (`Age`, `Amount`) with the **median**, which is robust to the outliers/contamination present.
- Filled missing categorical fields (`IncomeLevel`, `ProductCategory`, `Platform`, `Sentiment`) with `"Unknown"` instead of dropping rows, since missingness affected ~10% of records.
- Standardized text columns by stripping stray whitespace.
- Parsed dates that mixed `YYYY-MM-DD` and `DD/MM/YYYY` formats into a single consistent `datetime64` type.
- Cast ID columns to string for consistent joins/merges downstream.

**Challenges encountered**
- Each date column mixed two different formats, so a single-format parse silently produced ~100 missing dates per column.
- `Age` and `Amount` were stored as text because of embedded placeholder strings (`"Unknown"`, `"Free"`), blocking direct numeric conversion.
- Duplicate-ID counts were higher than duplicate full-row counts, meaning some "duplicate" records had slightly different values under the same ID.

**Decisions made**
- Categorical missing values -> `"Unknown"` category (preserves rows) vs. numeric missing values -> median (robust to skew/outliers).
- Deduplicate on the business key rather than relying on full-row equality, to catch near-duplicate records sharing the same ID.

**Observed patterns**
- Missingness in `IncomeLevel`, `Platform`, and `Sentiment` is concentrated around ~10% of rows, suggesting systematic non-response rather than random noise.
- Outlier ages (`-1`, `150`) and non-numeric amounts (`"Free"`) appear to be intentionally injected contamination for this exercise.